In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import pypsa
import xlsxwriter
import tz_pypsa
import tz_pypsa.wrangle as wrangle
import pandas as pd
# import tz_solve
import plotly.express as px
import plotly.graph_objects as go
from tz_pypsa.model import Model
from tz_pypsa.utils import get_examples
import os
import glob

In [1]:
# Path to the folder containing .nc files
folder_path = "C:/Users/jy/TransitionZero/Google - CFE - Documents/04. Country Specific Vault/Taiwan/2. Data & Results/Outputs/CFERun/TWN_P1_002/solved_networks/hourly_matching_CFE100_2030.nc"

# Initialize empty DataFrames for concatenation
curtailment_all = pd.DataFrame()
ci_curtailment_all = pd.DataFrame()

# Loop through all .nc files
for filename in os.listdir(folder_path):
    if filename.endswith(".nc"):
        full_path = os.path.join(folder_path, filename)
        file_key = os.path.splitext(filename)[0]  # strip .nc

        # Load the network
        n = pypsa.Network()
        n.import_from_netcdf(full_path)

        # Compute curtailment
        curtailment = (
            (
                n.generators_t.p_max_pu 
                * n.generators.p_nom_opt
                - n.generators_t.p
            )
            .dropna(axis=1, thresh=5)
        )

        # Rename columns to include file key (avoid duplicates)
        curtailment.columns = [f"{col}_{file_key}" for col in curtailment.columns]

        # Add to wide curtailment DataFrame
        curtailment_all = pd.concat([curtailment_all, curtailment], axis=1)

        # Compute C&I curtailment
        ci = n.generators.loc[n.generators.index.str.contains('C&I')]
        ci_curtailment = curtailment[[col for col in curtailment.columns if col.split("_")[0] in ci.index]]

        # Add to wide C&I curtailment DataFrame
        ci_curtailment_all = pd.concat([ci_curtailment_all, ci_curtailment], axis=1)

NameError: name 'pd' is not defined

In [42]:
onshorewind = ci_curtailment_all.filter(regex='onshorewind')
solar = ci_curtailment_all.filter(regex='solar')

In [44]:
px.line(
    onshorewind,
    x=onshorewind.index,
    y= onshorewind.columns,
    title='Onshore Wind Hourly Curtailment by Scenarios'
)

In [43]:
px.line(
    solar,
    x=solar.index,
    y=solar.columns,
    title='Solar Hourly Curtailment by Scenarios'
)